# Train Face Attribute Model

Lightweight multi-head classifier for the CelebA-derived face fields: `gender`, `face_fullness`, `cheekbones`, and `hairline`.

In [1]:
import json
from pathlib import Path
import sys
import time

import pandas as pd

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'backend').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not locate project root.')

def format_seconds(seconds: float) -> str:
    total_seconds = max(0, int(seconds))
    hours, remainder = divmod(total_seconds, 3600)
    minutes, secs = divmod(remainder, 60)
    if hours:
        return f'{hours:d}:{minutes:02d}:{secs:02d}'
    return f'{minutes:02d}:{secs:02d}'

def resolve_training_device(torch_module, require_gpu: bool = True) -> str:
    if torch_module.cuda.is_available():
        return 'cuda'
    if require_gpu:
        raise RuntimeError(
            'CUDA GPU is required for this training run, but the current PyTorch build does not have CUDA available. '
            'Install a CUDA-enabled PyTorch build into .venv and restart the notebook kernel.'
        )
    return 'cpu'

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
BACKEND_ROOT = PROJECT_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.append(str(BACKEND_ROOT))

import torch
from torch.utils.data import DataLoader

from app.ml.datasets import MultiAttributeDataset, read_jsonl_manifest
from app.ml.hairstyle_attribute_model import build_attribute_model, multitask_cross_entropy
from app.ml.metrics import attribute_accuracy, exact_match_accuracy
from app.ml.transforms import ResizeImage

PROJECT_ROOT

WindowsPath('D:/Projects/Personal Projects/Hairstyle Recommender Live Tryon')

In [2]:
DATASET_ROOT = PROJECT_ROOT / 'backend' / 'data' / 'datasets' / 'celeba_face_basic'
TRAIN_MANIFEST = DATASET_ROOT / 'train.jsonl'
VAL_MANIFEST = DATASET_ROOT / 'val.jsonl'
VOCAB_PATH = DATASET_ROOT / 'label_vocab.json'
SUMMARY_PATH = DATASET_ROOT / 'summary.json'
CHECKPOINT_PATH = PROJECT_ROOT / 'backend' / 'models' / 'celeba_face_basic' / 'face_attribute_model.pt'

REQUIRE_GPU = True
IMAGE_SIZE = 224
BATCH_SIZE = 16
EPOCHS = 5
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 0
DEVICE = resolve_training_device(torch, require_gpu=REQUIRE_GPU)

FACE_FIELDS = ('gender', 'face_fullness', 'cheekbones', 'hairline')


In [3]:
train_records = read_jsonl_manifest(TRAIN_MANIFEST)
val_records = read_jsonl_manifest(VAL_MANIFEST)
label_vocab = json.loads(VOCAB_PATH.read_text(encoding='utf-8'))
summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))

attribute_vocab_sizes = {field: len(label_vocab[field]) for field in FACE_FIELDS}

print('Device:', DEVICE)
print('Torch version:', torch.__version__)
print('CUDA version:', torch.version.cuda)
print('Face fields:', FACE_FIELDS)
print('Train records:', len(train_records))
print('Val records:', len(val_records))
summary

Device: cpu
Face fields: ('gender', 'face_fullness', 'cheekbones', 'hairline')
Train records: 2000
Val records: 400


{'face_fields': ['gender', 'face_fullness', 'cheekbones', 'hairline'],
 'total_records': 2800,
 'train_records': 2000,
 'val_records': 400,
 'test_records': 400,
 'field_value_counts': {'gender': {'female': 1709, 'male': 1091},
  'face_fullness': {'slim': 2673, 'full': 127},
  'cheekbones': {'soft': 1524, 'high': 1276},
  'hairline': {'regular': 2596, 'receding': 204}}}

In [4]:
transform = ResizeImage((IMAGE_SIZE, IMAGE_SIZE))

train_dataset = MultiAttributeDataset(
    train_records,
    label_vocab=label_vocab,
    transform=transform,
    fields=FACE_FIELDS,
)
val_dataset = MultiAttributeDataset(
    val_records,
    label_vocab=label_vocab,
    transform=transform,
    fields=FACE_FIELDS,
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

batch = next(iter(train_loader))
batch['image'].shape, {field: tensor.shape for field, tensor in batch['labels'].items()}

(torch.Size([16, 3, 224, 224]),
 {'gender': torch.Size([16]),
  'face_fullness': torch.Size([16]),
  'cheekbones': torch.Size([16]),
  'hairline': torch.Size([16])})

In [5]:
model = build_attribute_model(attribute_vocab_sizes=attribute_vocab_sizes, base_channels=32, dropout=0.2).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)

sum(parameter.numel() for parameter in model.parameters())

1175272

In [6]:
def move_targets_to_device(targets, device):
    return {field: tensor.to(device) for field, tensor in targets.items()}

def filter_outputs(outputs, fields):
    return {field: outputs[field] for field in fields}

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    running_loss = 0.0
    total_batches = 0
    for batch in loader:
        images = batch['image'].to(device)
        targets = move_targets_to_device(batch['labels'], device)
        optimizer.zero_grad(set_to_none=True)
        outputs = model(images)
        outputs = filter_outputs(outputs, FACE_FIELDS)
        loss = multitask_cross_entropy(outputs, targets)
        loss.backward()
        optimizer.step()
        running_loss += float(loss.item())
        total_batches += 1
    return running_loss / max(total_batches, 1)

def evaluate(model, loader, device):
    model.eval()
    running_loss = 0.0
    total_batches = 0
    output_batches = []
    target_batches = []

    with torch.no_grad():
        for batch in loader:
            images = batch['image'].to(device)
            targets = move_targets_to_device(batch['labels'], device)
            outputs = model(images)
            outputs = filter_outputs(outputs, FACE_FIELDS)
            loss = multitask_cross_entropy(outputs, targets)
            running_loss += float(loss.item())
            total_batches += 1
            output_batches.append({field: tensor.detach().cpu() for field, tensor in outputs.items()})
            target_batches.append({field: tensor.detach().cpu() for field, tensor in targets.items()})

    merged_outputs = {
        field: torch.cat([batch[field] for batch in output_batches], dim=0)
        for field in FACE_FIELDS
    }
    merged_targets = {
        field: torch.cat([batch[field] for batch in target_batches], dim=0)
        for field in FACE_FIELDS
    }
    attr_scores = attribute_accuracy(merged_outputs, merged_targets)
    return {
        'val_loss': running_loss / max(total_batches, 1),
        'exact_match_accuracy': exact_match_accuracy(merged_outputs, merged_targets),
        **attr_scores,
    }


In [7]:
history = []
best_metric = float('-inf')
best_epoch = None
epoch_durations = []

for epoch in range(1, EPOCHS + 1):
    print(f'Running epoch {epoch}/{EPOCHS}...')
    epoch_start = time.time()
    train_loss = train_one_epoch(model, train_loader, optimizer, DEVICE)
    metrics = evaluate(model, val_loader, DEVICE)
    epoch_duration = time.time() - epoch_start
    epoch_durations.append(epoch_duration)

    row = {
        'epoch': epoch,
        'train_loss': train_loss,
        'epoch_seconds': round(epoch_duration, 2),
        **metrics,
    }
    history.append(row)

    current_metric = metrics['exact_match_accuracy']
    if current_metric > best_metric:
        best_metric = current_metric
        best_epoch = epoch

    average_epoch_seconds = sum(epoch_durations) / len(epoch_durations)
    remaining_seconds = average_epoch_seconds * (EPOCHS - epoch)

    CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    torch.save(
        {
            'model_state_dict': model.state_dict(),
            'history': history,
            'last_epoch': epoch,
            'core_fields': FACE_FIELDS,
            'label_vocab': label_vocab,
        },
        CHECKPOINT_PATH,
    )

    print(f"Completed epoch {epoch}/{EPOCHS}")
    print(f"Epoch time: {format_seconds(epoch_duration)}")
    print(f"Estimated remaining: {format_seconds(remaining_seconds)}")
    print(f"Best epoch so far: {best_epoch} (exact_match_accuracy={best_metric:.4f})")
    print(row)
    display(pd.DataFrame(history))

print(f'Saved checkpoint to {CHECKPOINT_PATH}')
pd.DataFrame(history)

Running epoch 1/5...
Completed epoch 1/5
Epoch time: 00:39
Estimated remaining: 02:38
Best epoch so far: 1 (exact_match_accuracy=0.3575)
{'epoch': 1, 'train_loss': 1.84839741897583, 'epoch_seconds': 39.67, 'val_loss': 1.78270583152771, 'exact_match_accuracy': 0.35749998688697815, 'gender': 0.6424999833106995, 'face_fullness': 0.9549999833106995, 'cheekbones': 0.5874999761581421, 'hairline': 0.9150000214576721}


,epoch,train_loss,epoch_seconds,val_loss,exact_match_accuracy,gender,face_fullness,cheekbones,hairline
0,1,1.848397,39.67,1.782706,0.3575,0.6425,0.955,0.5875,0.915


Running epoch 2/5...
Completed epoch 2/5
Epoch time: 00:35
Estimated remaining: 01:52
Best epoch so far: 1 (exact_match_accuracy=0.3575)
{'epoch': 2, 'train_loss': 1.7332327213287353, 'epoch_seconds': 35.64, 'val_loss': 1.7370418739318847, 'exact_match_accuracy': 0.3100000023841858, 'gender': 0.6775000095367432, 'face_fullness': 0.9549999833106995, 'cheekbones': 0.5575000047683716, 'hairline': 0.9150000214576721}


,epoch,train_loss,epoch_seconds,val_loss,exact_match_accuracy,gender,face_fullness,cheekbones,hairline
0,1,1.848397,39.67,1.782706,0.3575,0.6425,0.955,0.5875,0.915
1,2,1.733233,35.64,1.737042,0.3100,0.6775,0.955,0.5575,0.915


Running epoch 3/5...
Completed epoch 3/5
Epoch time: 00:33
Estimated remaining: 01:12
Best epoch so far: 3 (exact_match_accuracy=0.3900)
{'epoch': 3, 'train_loss': 1.6734551801681519, 'epoch_seconds': 33.2, 'val_loss': 1.6378356742858886, 'exact_match_accuracy': 0.38999998569488525, 'gender': 0.7850000262260437, 'face_fullness': 0.9549999833106995, 'cheekbones': 0.512499988079071, 'hairline': 0.9150000214576721}


...

,epoch,train_loss,epoch_seconds,val_loss,exact_match_accuracy,gender,face_fullness,cheekbones,hairline
0,1,1.848397,39.67,1.782706,0.3575,0.6425,0.955,0.5875,0.915
1,2,1.733233,35.64,1.737042,0.3100,0.6775,0.955,0.5575,0.915
2,3,1.673455,33.20,1.637836,0.3900,0.7850,0.955,0.5125,0.915
3,4,1.564155,33.92,1.600812,0.4175,0.7350,0.955,0.6200,0.915


Running epoch 5/5...
Completed epoch 5/5
Epoch time: 00:32
Estimated remaining: 00:00
Best epoch so far: 4 (exact_match_accuracy=0.4175)
{'epoch': 5, 'train_loss': 1.4678456654548646, 'epoch_seconds': 32.76, 'val_loss': 1.6330592250823974, 'exact_match_accuracy': 0.4025000035762787, 'gender': 0.7699999809265137, 'face_fullness': 0.9549999833106995, 'cheekbones': 0.5824999809265137, 'hairline': 0.9150000214576721}


,epoch,train_loss,epoch_seconds,val_loss,exact_match_accuracy,gender,face_fullness,cheekbones,hairline
0,1,1.848397,39.67,1.782706,0.3575,0.6425,0.955,0.5875,0.915
1,2,1.733233,35.64,1.737042,0.3100,0.6775,0.955,0.5575,0.915
2,3,1.673455,33.20,1.637836,0.3900,0.7850,0.955,0.5125,0.915
3,4,1.564155,33.92,1.600812,0.4175,0.7350,0.955,0.6200,0.915
4,5,1.467846,32.76,1.633059,0.4025,0.7700,0.955,0.5825,0.915


Saved checkpoint to D:\Projects\Personal Projects\Hairstyle Recommender Live Tryon\backend\checkpoints\celeba_face_basic\face_attribute_model.pt


,epoch,train_loss,epoch_seconds,val_loss,exact_match_accuracy,gender,face_fullness,cheekbones,hairline
0,1,1.848397,39.67,1.782706,0.3575,0.6425,0.955,0.5875,0.915
1,2,1.733233,35.64,1.737042,0.3100,0.6775,0.955,0.5575,0.915
2,3,1.673455,33.20,1.637836,0.3900,0.7850,0.955,0.5125,0.915
3,4,1.564155,33.92,1.600812,0.4175,0.7350,0.955,0.6200,0.915
4,5,1.467846,32.76,1.633059,0.4025,0.7700,0.955,0.5825,0.915
